# CRN Surrogate: Mass-Action 3s Experiment

Run the full mass-action experiment (data generation + training + evaluation) on Colab GPU.

**What this does:**
1. Clones the repo and installs the package
2. Generates a dataset of random mass-action CRNs (1-3 species, 2-6 reactions)
3. Trains the encoder + neural SDE on the dataset
4. Evaluates on novel random topologies


In [ ]:
# --- Colab setup ---
import subprocess, sys, os

REPO_URL = "https://github.com/Mijan/CRN_Surrogate.git"
CLONE_DIR = "/content/CRN_Surrogate"
SRC_DIR = "/content/CRN_Surrogate/src"
BRANCH = "main"  # change if working on a feature branch

if not os.path.exists(CLONE_DIR):
    print("Cloning repo...")
    subprocess.check_call(["git", "clone", "-q", "-b", BRANCH, REPO_URL, CLONE_DIR])
else:
    print(f"Using existing clone at {CLONE_DIR}")
    # Pull latest changes
    subprocess.check_call(["git", "-C", CLONE_DIR, "pull", "-q"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", CLONE_DIR])

os.chdir(CLONE_DIR)
print(f"Working directory: {os.getcwd()}")

# Verify GPU
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Training will be slow.")


## W&B Authentication

Run the cell below to log in. You can find your API key at https://wandb.ai/authorize.

If you don't want W&B logging, set `USE_WANDB = False` and skip this cell.

In [ ]:
USE_WANDB = True

if USE_WANDB:
    import wandb
    wandb.login()


## Configuration

Review the experiment config before running. Adjust if needed (e.g. reduce `n_train` for a quick test run).

In [ ]:
sys.path.insert(0, CLONE_DIR)
sys.path.insert(0, SRC_DIR)
from experiments.configs.mass_action_3s import MassAction3sV3Config
from experiments.configs.registry import get_config

cfg = get_config("mass_action_3s_v3")
print(f"Experiment: {cfg.experiment_name}")
print(f"Max species: {cfg.max_n_species}, Max reactions: {cfg.max_n_reactions}")
print(f"d_model: {cfg.d_model}, d_hidden: {cfg.d_hidden}")
print(f"Dropout: context_dropout={cfg.context_dropout}, mlp_dropout={cfg.mlp_dropout}")
print(f"Training: {cfg.max_epochs} epochs, batch_size={cfg.batch_size}, lr={cfg.lr}, val_every={cfg.val_every}")
print(f"Dataset: {cfg.dataset.n_train} train, {cfg.dataset.n_val} val")
print(f"  SSA trajectories per CRN: {cfg.dataset.n_ssa_trajectories}")
print(f"  Time grid: [0, {cfg.dataset.t_max}] with {cfg.dataset.n_time_points} points")


## Step 1: Generate Dataset

This samples random mass-action CRNs, simulates SSA trajectories, curates, and saves.
Takes ~5-15 minutes depending on `n_train`.

In [ ]:
wandb_flag = "" if USE_WANDB else "--no-wandb"

!python -u experiments/scripts/generate_dataset.py \
    --config mass_action_3s_v3 \
    --output-dir experiments/datasets \
    {wandb_flag} \
    --seed 42 \
    --checkpoint-every 500 \
    --sim-timeout 30 \
    --n-init-conditions 4


### Quick data inspection

In [ ]:
import json
from pathlib import Path
import wandb

# meta_path = Path("experiments/datasets") / f"{cfg.experiment_name}_meta.json"
# meta = json.loads(meta_path.read_text())

api = wandb.Api()
artifact = api.artifact("jan-mikelson-independent/crn-surrogate/mass_action_3s_v7_dataset:latest")
artifact_dir = Path(artifact.download())

# Load metadata
meta = json.loads((artifact_dir / "mass_action_3s_v3_meta.json").read_text())

print(f"Train items: {meta['n_train']}")
print(f"Val items: {meta['n_val']}")
print(f"Train pass rate: {meta['train_meta']['pass_rate']:.0%}")
print(f"Val pass rate: {meta['val_meta']['pass_rate']:.0%}")
print(f"Species distribution (train): {meta['train_meta']['n_species_dist']}")
print(f"Reactions distribution (train): {meta['train_meta']['n_reactions_dist']}")


## Step 2: Train

Runs the training script with GPU. Progress is logged to W&B.

**For a quick test**, override `--max-epochs 20`. For the full run, remove the override.

**Auto-resume:** If the Colab runtime disconnects, re-run this cell.
The `--resume auto` flag automatically downloads the latest checkpoint
from W&B and continues training from where it left off. The optimizer
state and learning rate schedule are fully restored.

In [ ]:
!python -u experiments/scripts/train.py \
    --config mass_action_3s_v5 \
    --device cuda \
    --wandb-artifact "mass_action_3s_v3_dataset:latest" \
    --seed 42 \
    --resume auto

## Step 3: Evaluate on Novel Topologies

Load the trained model and compare against SSA on CRNs the model has never seen.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from pathlib import Path

from crn_surrogate.configs.model_config import ModelConfig
from crn_surrogate.data.generation.mass_action_generator import MassActionCRNGenerator
from crn_surrogate.encoder.bipartite_gnn import BipartiteGNNEncoder
from crn_surrogate.encoder.tensor_repr import crn_to_tensor_repr
from crn_surrogate.evaluation import ModelEvaluator, TrajectoryComparator
from crn_surrogate.simulation import GillespieSSA, FastMassActionSSA
from crn_surrogate.simulation.trajectory import Trajectory
from crn_surrogate.simulator.neural_sde import CRNNeuralSDE
from crn_surrogate.simulator.sde_solver import EulerMaruyamaSolver
from experiments.configs.registry import get_config

cfg = get_config("mass_action_3s_v8")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Try local checkpoint first, then W&B
ckpt_dir = Path("checkpoints")
ckpt_files = sorted(ckpt_dir.glob("best_epoch*.pt")) if ckpt_dir.exists() else []

if ckpt_files:
    ckpt_path = ckpt_files[-1]
    print(f"Loading local checkpoint: {ckpt_path}")
else:
    print("No local checkpoint found, downloading from W&B...")
    import wandb
    api = wandb.Api()
    artifact = api.artifact(
                f"{cfg.wandb_project}/{cfg.experiment_name}_train_periodic_checkpoint:latest",
    )
    artifact_dir = Path(artifact.download())
    ckpt_path = sorted(artifact_dir.glob("*.pt"))[-1]
    print(f"Downloaded: {ckpt_path}")

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

model_config = cfg.build_model_config()
encoder = BipartiteGNNEncoder(model_config.encoder).to(device)
sde = CRNNeuralSDE(model_config.sde, n_species=cfg.max_n_species).to(device)
encoder.load_state_dict(ckpt["encoder_state"])
sde.load_state_dict(ckpt["sde_state"])
encoder.eval()
sde.eval()
print("Model loaded.")

No local checkpoint found, downloading from W&B...


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/jan/.netrc.
wandb:   1 of 1 files downloaded.  


Downloaded: /home/jan/Dropbox/personal_projects/CRN_Surrogate/experiments/analysis/artifacts/mass_action_3s_v8_train_periodic_checkpoint:v0/periodic_epoch10.pt
Model loaded.


In [ ]:
# Generate novel test CRNs
# torch.manual_seed(53)  # different seed from training
torch.manual_seed(24)# different seed from training
gen = MassActionCRNGenerator(cfg.dataset.generator)
ssa_fast = FastMassActionSSA()
solver = EulerMaruyamaSolver(model_config.sde)


topologies = gen._topology_sampler.sample_batch(8)
test_crns = [gen.sample_from_topology(t) for t in topologies]
# test_crns = gen.sample_batch(8)
time_grid = torch.linspace(0.0, cfg.dataset.t_max, cfg.dataset.n_time_points)
K_EVAL, M_EVAL = 30, 30

fig, axes = plt.subplots(len(test_crns), 3, figsize=(16, 4 * len(test_crns)))
if len(test_crns) == 1:
    axes = axes[None, :]  # ensure 2D

for row, crn in enumerate(test_crns):
    crn_repr = crn_to_tensor_repr(crn).to(device)
    init_state = gen.sample_initial_state(crn)
    n_actual = crn.n_species

    fast_input = ssa_fast.extract_topology_arrays(crn)
    ssa_tensor = ssa_fast.simulate_batch(stoichiometry=fast_input["stoichiometry"], reactant_matrix=fast_input["reactant_matrix"], rate_constants=fast_input["rate_constants"], initial_state=init_state,  t_max= cfg.dataset.t_max,
        time_grid=time_grid,
        n_trajectories=M_EVAL)

    # Neural SDE rollouts (with padding)
    with torch.no_grad():
        ctx = encoder(crn_repr)
        padded_init = torch.zeros(cfg.max_n_species, device=device)
        padded_init[:n_actual] = init_state.to(device)

        sde_samples = []
        for _ in range(K_EVAL):
            traj = solver.solve(sde, padded_init, ctx, time_grid.to(device), dt=cfg.dt)
            sde_samples.append(traj.states[:, :n_actual].cpu())
        sde_tensor = torch.stack(sde_samples)

    # Plot first 3 species (or fewer)
    t_np = time_grid.numpy()
    n_plot = min(n_actual, 3)
    for s in range(n_plot):
        ax = axes[row, s]
        # SSA
        for m in range(min(M_EVAL, 10)):
            ax.plot(t_np, ssa_tensor[m, :, s].numpy(), color="gray", alpha=0.2, lw=0.5)
        ax.plot(t_np, ssa_tensor[:, :, s].mean(dim=0).numpy(),
                color="black", lw=2, label="SSA mean")
        # SDE
        for k in range(min(K_EVAL, 10)):
            ax.plot(t_np, sde_tensor[k, :, s].numpy(), color="steelblue", alpha=0.2, lw=0.5)
        ax.plot(t_np, sde_tensor[:, :, s].mean(dim=0).numpy(),
                color="steelblue", lw=2, label="SDE mean")

        if row == 0:
            ax.set_title(f"Species {s}")
        if s == 0:
            ax.set_ylabel(f"CRN {row}\n({n_actual}sp, {crn.n_reactions}rxn)")
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    # Blank unused columns
    for s in range(n_plot, 3):
        axes[row, s].axis("off")

plt.suptitle("Neural SDE vs SSA on Novel Topologies", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Summary metrics
print(f"{'CRN':>4} {'n_sp':>5} {'n_rxn':>6} {'mean_mse':>10} {'var_mse':>10}")
print("-" * 40)

for i, crn in enumerate(test_crns):
    crn_repr = crn_to_tensor_repr(crn).to(device)
    init_state = gen.sample_initial_state(crn)
    n_actual = crn.n_species

    ssa_trajs_list = ssa.simulate_batch(
        stoichiometry=crn.stoichiometry_matrix,
        propensity_fn=crn.evaluate_propensities,
        initial_state=init_state,
        t_max=cfg.dataset.t_max,
        n_trajectories=M_EVAL,
    )
    ssa_tensor = Trajectory.stack_on_grid(ssa_trajs_list, time_grid)

    with torch.no_grad():
        ctx = encoder(crn_repr)
        padded_init = torch.zeros(cfg.max_n_species, device=device)
        padded_init[:n_actual] = init_state.to(device)
        sde_samples = []
        for _ in range(K_EVAL):
            traj = solver.solve(sde, padded_init, ctx, time_grid.to(device), dt=cfg.dt)
            sde_samples.append(traj.states[:, :n_actual].cpu())
        sde_tensor = torch.stack(sde_samples)

    sde_mean = sde_tensor.mean(dim=0)
    ssa_mean = ssa_tensor.mean(dim=0)
    mean_mse = ((sde_mean - ssa_mean) ** 2).mean().item()

    sde_var = sde_tensor.var(dim=0)
    ssa_var = ssa_tensor.var(dim=0)
    var_mse = ((sde_var - ssa_var) ** 2).mean().item()

    print(f"{i:>4} {n_actual:>5} {crn.n_reactions:>6} {mean_mse:>10.4f} {var_mse:>10.4f}")

In [ ]:
if USE_WANDB:
    import wandb
    eval_run = wandb.init(
        project=cfg.wandb_project,
        group=cfg.wandb_group,
        job_type="evaluation",
        name=f"{cfg.experiment_name}_eval",
    )
    # After computing metrics:
    for i, (crn, mean_mse, var_mse) in enumerate(results):
        eval_run.log({"test_crn": i, "mean_mse": mean_mse, "var_mse": var_mse})
    # Log the comparison figure:
    eval_run.log({"trajectory_comparison": wandb.Image(fig)})
    eval_run.finish()

### Test CRN topologies

In [ ]:
from crn_surrogate.data.generation.mass_action_topology import MassActionTopology

for i, (topo, crn) in enumerate(zip(topologies, test_crns)):
    print(f"=== CRN {i} ===")
    print(topo.summary())


### Training curves (from checkpoint)

In [ ]:
if "train_losses" in ckpt:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    losses = ckpt["train_losses"]
    epochs = range(1, len(losses) + 1)

    axes[0].plot(epochs, losses, color="steelblue", lw=1.5)
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Train Loss (log)")
    axes[0].set_title("Training Loss")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, losses, color="steelblue", lw=1.5)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Train Loss")
    axes[1].set_title("Training Loss (linear)")
    axes[1].grid(True, alpha=0.3)

    if "val_losses" in ckpt and ckpt["val_losses"]:
        val_losses = ckpt["val_losses"]
        val_epochs = list(range(cfg.val_every, cfg.val_every * len(val_losses) + 1, cfg.val_every))
        for ax in axes:
            ax.plot(val_epochs[:len(val_losses)], val_losses, color="tomato",
                    lw=1.5, marker="o", ms=3, label="Val")
            ax.legend()

    plt.tight_layout()
    plt.show()
else:
    print("No training losses in checkpoint.")
